# Visualisation interactive des résultats EF d'Hermite

Ce notebook présente le module `affichage_interactif` (Plotly + ipywidgets), qui complète `pb.Affichage(...)` :

- **survol à la souris** : N, T, M, K, |u| en tout point d'un élément ; ddls et réactions aux noeuds ;
- **contrôles** : champ affiché, couleur sur la déformée ou *diagramme* (N, T, M), amplification, numéros ;
- **déformée interpolée par les fonctions d'Hermite** (cubiques), comparée aux segments droits entre noeuds ;
- **modes propres animés** (flambage, vibrations) ; **superposition** de maillages pour la convergence.

Zoom : cadre à la souris ; double-clic pour revenir. Export PNG : icône appareil photo.

In [ ]:
import sys
sys.path.append("exemples/")
import numpy as np
import Hermitte_NL_Classes as Cl
import affichage_interactif as AI

# Section rectangulaire en aluminium
E, rho = 70.E9, 2.6E3
bs, hs = 0.15, 0.3
ES, EI, Smasse = E*bs*hs, E*bs*hs**3/12, rho*bs*hs

def console(L, Nl, forces=(), angle=0.):
    """Console encastrée au noeud 1, de longueur L, maillée avec Nl éléments."""
    nodes_global, nodes_elem = Cl.fonct_Maille_Poutre(L, Nl, angle)
    props_elem = [np.repeat([[ES, EI, Smasse]], Nl, axis=0), np.zeros((Nl, 2))]
    pb = Cl.pbEF_poutre(nodes_global, nodes_elem, props_elem)
    pb.Maillage_Assemblage()
    pb.CL_forces = [list(f) for f in forces]
    pb.CL_deplacements = [[1, 1, 0], [1, 2, 0], [1, 3, 0]]
    pb.Application_CL()
    return pb

## 1. Console chargée en bout, avec **un seul élément**

La solution exacte $v(x) = \dfrac{P x^2 (3L - x)}{6EI}$ est un polynôme de degré 3 : elle est **exactement** représentée par l'interpolation d'Hermite.

- ➡ Décocher *Interpolation d'Hermite* : avec des segments droits, la déformée d'un seul élément paraît fausse alors que la solution EF est exacte.
- ➡ Passer le champ à *M* puis à *T* : $M$ est linéaire dans l'élément, $T$ constant.

In [ ]:
L, P = 2., -1.E3
pb1 = console(L, 1, forces=[[2, 2, P]])
pb1.Resol()
pb1.Calcul_Def_Eff()
pb1.Calcul_Reactions()
print(f"Flèche EF = {pb1.nodes_global[-1].u2:.6e} m   ;   PL³/3EI = {P*L**3/(3*EI):.6e} m")
AI.vue(pb1)

## 2. Portique (maillage lu dans des fichiers)

Mât de 3 m encastré à sa base et potence de 2 m, effort vertical en bout et effort horizontal en tête de mât.

- ➡ Choisir *Moment fléchissant M* puis la vue **Diagramme** : on retrouve le diagramme des moments de la RdM. Les valeurs positives sont tracées du côté de la normale locale $\underline{n}$ (vecteur tangent $\underline{t}$ tourné de +90°, $\underline{t}$ orienté du noeud 1 vers le noeud 2 de l'élément).
- ➡ Survoler un appui pour lire les réactions.

In [ ]:
nodes_global, nodes_elem = Cl.fonct_Maille_depuis_Fichier('exemples/portique_noeuds.txt', 'exemples/portique_elem.txt')
nb_elem = len(nodes_elem)
props_elem = [np.repeat([[ES, EI, Smasse]], nb_elem, axis=0), np.zeros((nb_elem, 2))]
pp = Cl.pbEF_poutre(nodes_global, nodes_elem, props_elem)
pp.Maillage_Assemblage()
pp.CL_forces = [[11, 2, -5.E3], [7, 1, 2.E3]]
pp.CL_deplacements = [[1, 1, 0], [1, 2, 0], [1, 3, 0]]
pp.Application_CL()
pp.Resol()
pp.Calcul_Def_Eff()
pp.Calcul_Reactions()
AI.vue(pp, champ='M')

## 3. Modes propres de vibration

- ➡ Choisir le mode, puis **▶ Animer** dans la figure. Le 3ᵉ mode de cette console courte est le premier mode **axial**.
- ➡ Avec peu d'éléments, décocher *Interpolation d'Hermite* : les modes élevés deviennent méconnaissables avec des segments droits.

In [ ]:
pv = console(2., 4)
pv.Calcul_Modes_Propres('Vibration', 6)
AI.vue_modes(pv)

## 4. Convergence en maillage : superposition

Même mode calculé avec 1, 2, 4 et 8 éléments. Les pulsations sont rappelées sous le titre : la masse cohérente donne des valeurs **par excès**, décroissantes avec le raffinement.

In [ ]:
maillages = [console(2., n) for n in (1, 2, 4, 8)]
for p in maillages:
    p.Calcul_Modes_Propres('Vibration', 3)
noms = ['1 élt', '2 élts', '4 élts', '8 élts']
AI.compare(maillages, noms, mode=2).show()
AI.compare(maillages, noms, mode=2, hermite=False).show()

## 5. Flambage

Console comprimée par un effort unitaire en bout : la charge critique théorique est $\pi^2 EI / 4L^2$.

In [ ]:
pf = console(2., 6, forces=[[7, 1, -1.]])
pf.Resol()
pf.Calcul_Def_Eff()
pf.MAJ_Rigidite_Geom()
pf.Calcul_Modes_Propres('Flambage', 4)
print(f"Euler : {np.pi**2*EI/(4*2.**2):.4e} N")
AI.vue_modes(pf)

## 6. Sans widgets : figures seules, export HTML

Les fonctions `AI.figure`, `AI.figure_mode` et `AI.compare` renvoient des figures Plotly. Un fichier HTML exporté reste interactif (survol, zoom, animation) et peut être déposé sur un ENT ou envoyé aux étudiants.

In [ ]:
fig = AI.figure(pp, 'T', representation='diagramme', numeros=True)
fig.show()
# fig.write_html("portique_effort_tranchant.html")